# သင်ခန်းစာ ၀၁ - AI များအဖော်ပြု ကိုယ်စားလှယ်များ အကြောင်းသိရှိခြင်း

**AI ကိုယ်စားလှယ်များအတွက် စတင်လေ့လာသူများ** သင်တန်း၏ ပထမဆုံးသင်ခန်းစာကိုကြိုဆိုပါတယ်!

**AI ကိုယ်စားလှယ်** သည် အကြီးစားဘာသာစကားမော်ဒယ် (LLM) ကို သဘောသယံဇာတ အင်ဂျင်အဖြစ် အသုံးပြုကာ အသုံးပြုသူအတွက် ရည်မှန်းချက်တစ်ခု ပြည့်မှီစေဖို့ အစစ်အမှန်ကမ္ဘာထဲတွင် ကိရိယာများကိုခေါ်ဆိုခြင်း၊ ဒေတာဘေ့စ် မေးမြန်းခြင်း သို့မဟုတ် ကုဒ်ဆောင်ရွက်ခြင်း စသဖြင့် *လုပ်ဆောင်ချက်များ* ဆောင်ရွက်နိုင်သော ပရိုဂရမ်တစ်ခုဖြစ်သည်။

ဤစာအုပ်တွင် သင်သည် သင်၏ ပထမဆုံး ကိုယ်စားလှယ်တစ်ဦးဖြစ်သော **ခရီးသွား ကိုယ်စားလှယ်** ကို တည်ဆောက်မှာဖြစ်ပြီး ခရီးသွားရာ ဦးတည်ရာများ အကြံပြုပေးမည်ဖြစ်သည်။ လမ်းလျှောက်သည့်အချိန်တွင် သင်သည်:

1. **Microsoft Agent Framework** ကို အသုံးပြု၍ Azure AI Foundry ကိုယ်စားလှယ်ဝန်ဆောင်မှုနှင့် ချိတ်ဆက်နည်း။
2. ကိုယ်စားလှယ်အတွက် **ကိရိယာ** တစ်ခု ဖန်တီးပေးခြင်း — ပြုံးလှည့်သုံး Python ဖန်ရှင်တစ်ခု ဖြစ်ပါသည်။
3. ကိုယ်စားလှယ်ကို လုပ်ဆောင်ကာ ၎င်း၏ တုံ့ပြန်ချက်ကို စစ်ဆေးခြင်း။
4. ကိုယ်စားလှယ်၏ တုံ့ပြန်ချက်ကို တစ်အက္ခရာချင်းစီ စီးဆင်းဖော်ပြခြင်း။


## Setup

ဤ notebook ကို run မလုပ်မီ အောက်ပါအချက်များရှိရန် သေချာစေပါ။

1. **တပ်ဆင်ပြီးသား chat model (ဥပမာ `gpt-4o-mini`) ပါသော Azure AI Foundry project တစ်ခု** ရှိရန်။
2. **Azure CLI ဖြင့် log in ပြီးသား ဖြစ်ရန်** — သင်၏ terminal တွင် `az login` လုပ်ပါ။
3. **လိုအပ်သော environment variables များကို သတ်မှတ်ထားရန်။**
   - `AZURE_AI_PROJECT_ENDPOINT` — သင်၏ Azure AI Foundry project endpoint။
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — သင့် တပ်ဆင်ထားသော model အမည်။

အောက်ပါ cell တွင် သင်လိုအပ်သော Python packages များ တပ်ဆင်ပေးပါသည်။


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## သင်၏ ပထမဆုံး Agent ဖန်တီးခြင်း

Agent တစ်ခုတွင် လိုအပ်သည့် အရာနှစ်ခုရှိသည်-

- **ညွှန်ကြားချက်များ** - သူသည် *သူ့ဘာသာဘာဖြစ်သည်* နှင့် *ဘယ်လို အပြုအမူ ပြုလုပ်ရမည်* ဆိုသည်ကို ပြောပြသော (စနစ် prompt)။
- **ကိရိယာများ** — Agent သည် အချက်အလက် ရယူခြင်း သို့မဟုတ် လုပ်ဆောင်ချက်များ ပြုလုပ်ရာတွင် ခေါ်ယူနိုင်သော `@tool` ဖြင့် အမှတ်အသားပြုထားသော Python လုပ်ဆောင်ချက်များ။

အောက်တွင် လူကြိုက်များသော အကျဉ်းစား ခရီးစဉ် ဖြစ်ထွန်းရာများစာရင်းကို ပြန်လာသော ကိရိယာ လက်တွေ့ ဖေါ်ပြထားသည်။ အသုံးပြုသူသည် ခရီးသွားအကြံပြုချက်များ မေးမြန်းသောအခါ Agent သည် ဤကိရိယာကို အသုံးပြုမည်ဖြစ်သည်။


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

အပြန်အလှန် ဆက်သွယ်မှု ပိုမိုကောင်းမွန်စေရန် အတွက် မေးမြန်းသူ၏ တုံ့ပြန်ချက်ကို **စီးဆင်း** ပြသနိုင်သည်။ ပြည့်စုံသော အဖြေကို စောင့်ဆိုင်းရန်မလိုဘဲ၊ မေးမြန်းသူသည် စာသား အပိုင်းအစများကို ထုတ်ပြန်ပေးသည်။ ၎င်းသည် ချက်ချင်း အထွက်ကို ပြသလိုသော စကားဝိုင်း မျက်နှာပြင်များတွင် အလွန်အသုံးဝင်သည်။


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## အနှစ်ချုပ်

ဒီသင်ခန်းစာမှာ သင်တွေ့ရှိခဲ့တာက:

- `AzureAIProjectAgentProvider` ဖြင့် Azure AI Foundry Agent Service နဲ့ ချိတ်ဆက်ပေးတဲ့ **provider ကို ဖန်တီးခြင်း**။
- အေးဂျင့်က သင့် Python ဖင်ရှင်တွေ ခေါ်နိုင်ဖို့အတွက် `@tool` decorator ကို အသုံးပြု၍ **ကိရိယာတစ်ခုကို သတ်မှတ်ခြင်း**။
- အသုံးပြုသူမက်ဆေ့ချ်နဲ့ အေးဂျင့်ကို ပြေးပြီး ပြန်လည်ဖြေပေးမှုကို ပရင့်ထုတ်ခြင်း။
- အပိုင်းပိုင်း ဖြေကြားချက်များကို **ထံ့ထွင်းစမ်းသပ်ရိုက်ထွက်အနေနဲ့ စီးဆင်းထုတ်ပေးခြင်း**။

နောက်သင်ခန်းစာမှာတော့ agentic framework များကို နက်နဲစွာ လေ့လာပြီး အေးဂျင့်တွေကို ပို၍ အင်အားကြီးတဲ့ ကိရိယာများနှင့် အဆင့်မြင့် အတွေးအခေါ် ခွင့်ပြုချက်များ ပေးနိုင်ခြင်းကို လေ့လာပေးမှာ ဖြစ်ပါတယ်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**မှတ်ချက်**  
ဤစာရွက်စာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးစားနေသော်လည်း အလိုအလျောက်ဘာသာပြန်မှုများတွင် အမှားများ သို့မဟုတ် မှန်ကန်မှုလျော့ပါးမှုများ ဖြစ်ပေါ်နိုင်ကြောင်း သတိပြုပါမည်။ မူရင်းစာရွက်စာတမ်းကို မိမိဘာသာစကားဖြင့်သာ ထောက်ခံအာမခံရမည့် အချက်အလက်အဖြစ် သတ်မှတ်အသုံးပြုသင့်ပါသည်။ အရေးပါသောအချက်အလက်များအတွက် ကျွမ်းကျင်သူများမှ လက်ရှိဘာသာပြန်ခြင်းကို အကြံပြုပါသည်။ ဤဘာသာပြန်မှုအား အသုံးပြုပြီး ဖြစ်ပေါ်နိုင်သည့် နားမလည်မှုများ သို့မဟုတ် မမှန်ကန်သော နားလည်မှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မယူပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
